In [61]:
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import treeswift
import sys
import os
import csv
import copy

In [75]:
# Folder containing CSV embeddings
folder = Path("k7_v39_8k_s28_clade_All_TOL_Contigs_Global_logest/embedding")
tree_path = "k7_v39_8k_s28_clade_All_TOL_Contigs_Global_logest"

fastme_folder = "fastme"

# Read all CSV files and combine into a single dataframe
dfs = []
for f in folder.glob("*.emb"):
    df = pd.read_csv(f, index_col=None, header=None, sep='\t')
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)

#print(combined.head())


In [76]:
combined

,0,1,2,3,4,5,6,7,8,9,...,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024
0,G900094975.part_NZ_FMAU01000004.1,2.110610,-2.142116,1.743075,3.292292,0.317864,0.635217,0.960903,-0.810099,1.244828,...,1.166639,-1.169913,1.391732,-0.091695,-2.170909,1.175744,-0.140221,-1.892233,0.783476,0.686700
1,G002006995.part_NZ_MRUL01000011.1,2.996699,-1.316467,1.702463,3.166666,0.846937,0.093720,0.019569,-0.141875,1.379195,...,0.955235,-1.496366,1.032822,1.301930,-1.165597,1.480875,-0.277595,-1.633658,1.591854,0.686339
2,G001916215.part_MNRZ01000001.1,2.171200,-1.820156,1.018497,3.199118,0.505145,0.433650,0.798948,-0.292440,0.715027,...,1.709804,-1.203840,0.929243,0.188309,-1.920189,1.273606,-0.098541,-2.023484,1.382548,0.382626
3,G000336655.part_NZ_AOLZ01000076.1,2.434413,-2.299406,0.737358,3.086267,0.596625,1.102972,0.352979,-0.033723,0.992594,...,2.076242,-1.304467,0.453213,0.847488,-2.029152,1.170989,-0.199750,-1.901293,1.540813,-0.465054
4,G000768875.part_NZ_JRAO01000018.1,2.179699,-1.636013,0.602564,3.250830,0.813923,-0.241795,0.413496,-0.778063,0.200575,...,1.973108,-0.491821,-0.662421,0.237713,-1.686678,1.810089,0.730740,-2.588176,0.850790,-1.254452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2257,G001312505.part_NZ_BBDW01000009.1,1.647420,-1.196415,1.121824,3.726898,0.402853,0.432481,0.944846,-0.993668,0.343317,...,2.137996,-1.435003,1.151912,0.723634,-1.699707,0.524546,-0.688231,-2.028146,0.816490,0.925960
2258,G001767605.part_MELM01000044.1,1.073423,-2.036982,1.989191,3.707403,1.160379,1.007376,0.358383,-0.853465,0.742896,...,1.568864,-1.661783,1.036779,0.063228,-2.233975,0.785448,-0.311207,-1.979377,1.047654,0.898358
2259,G000436935.part_FR891761.1,1.840690,-1.272071,0.669404,3.746860,0.356067,1.260029,0.416437,-0.989730,0.791753,...,1.307882,-1.813035,0.463641,1.428256,-1.779758,0.880081,-0.739084,-2.012335,0.870691,0.910092
2260,G001570425.part_NZ_BCSX01000018.1,1.919179,-1.041099,0.597000,3.559634,0.513732,0.482399,-0.049112,-0.290812,0.308444,...,1.745555,-1.811974,0.337623,0.351171,-1.301831,1.649788,0.446610,-1.491037,1.835489,0.020948


In [45]:
# First column is IDs, rest are embeddings
ids = combined.iloc[:, 0]
embeddings = combined.iloc[:, 1:]

# Convert to PyTorch tensor (float32)
embeddings_tensor = torch.tensor(embeddings.values, dtype=torch.float32)

# Compute pairwise L2 distance matrix
pairwise_outputs = torch.cdist(embeddings_tensor, embeddings_tensor, p=2, compute_mode='donot_use_mm_for_euclid_dist')

pairwise_outputs2 = torch.square(pairwise_outputs)
#pairwise_outputs2 = pairwise_outputs
#pairwise_outputs3 = torch.div(pairwise_outputs2, 100.0)

 # Round distances < 1e10-6 to 0 so apples can handle such values
pairwise_outputs3 = torch.where(pairwise_outputs2 < 1.0e-6, torch.tensor(0, dtype=pairwise_outputs2.dtype),pairwise_outputs2)

In [46]:
pairwise_outputs3

tensor([[  0.0000, 331.9149, 180.6709,  ..., 271.1494, 324.0011, 277.4739],
        [331.9149,   0.0000, 369.0501,  ..., 396.1748, 420.2906, 357.0605],
        [180.6709, 369.0501,   0.0000,  ..., 276.0433, 349.5323, 310.6284],
        ...,
        [271.1494, 396.1748, 276.0433,  ...,   0.0000, 381.2148, 339.5221],
        [324.0011, 420.2906, 349.5323,  ..., 381.2148,   0.0000, 340.4506],
        [277.4739, 357.0605, 310.6284,  ..., 339.5221, 340.4506,   0.0000]])

In [47]:
# Convert to pandas DataFrame with IDs
dist_df = pd.DataFrame(pairwise_outputs3, index=ids, columns=ids)

In [48]:
dist_df.to_csv(os.path.join(tree_path, "pairwise_distances_kf2vec_off_emb.tsv"), index=True, header=True, sep = '\t')

In [49]:
df = copy.deepcopy(dist_df)

In [50]:
import pandas as pd
from itertools import combinations, product

# df is your pairwise distance matrix DataFrame
# Leaves labeled like "sample1.part_001", "sample2.part_010", etc.

# --- Step 1: group contigs by sample ID ---
sample_to_contigs = {}
for leaf in df.index:
    sample_id = leaf.split(".part")[0]  # extract sample ID
    sample_to_contigs.setdefault(sample_id, []).append(leaf)

# --- Step 2: compute within-sample distances ---
within_distances = []
records = []
for sample, contigs in sample_to_contigs.items():
    if len(contigs) < 2:
        continue  # no pairs to compute
    # all pairwise combinations of contigs in the same sample
    for a, b in combinations(contigs, 2):
        within_distances.append(df.loc[a, b])
        records.append({
            'Contig1': a,
            'Contig2': b,
            'Distance': df.loc[a, b],
            'Type': 'Within',
            'Sample1': sample,
            'Sample2': sample
        })

mean_within = sum(within_distances) / len(within_distances)
print("Mean within-sample distance:", mean_within)

# --- Step 3: compute across-sample distances ---
across_distances = []
samples = list(sample_to_contigs.keys())
for i in range(len(samples)):
    for j in range(i + 1, len(samples)):
        contigs_i = sample_to_contigs[samples[i]]
        contigs_j = sample_to_contigs[samples[j]]
        # all pairs between two different samples
        for a, b in product(contigs_i, contigs_j):
            across_distances.append(df.loc[a, b])
            records.append({
                'Contig1': a,
                'Contig2': b,
                'Distance': df.loc[a, b],
                'Type': 'Across',
                'Sample1': samples[i],
                'Sample2': samples[j]
            })

mean_across = sum(across_distances) / len(across_distances)
print("Mean across-sample distance:", mean_across)

# --- Step 4: save to text file ---
output_df = pd.DataFrame(records)

all_distances = output_df['Distance'].values
min_val = all_distances.min()
max_val = all_distances.max()
output_df['Distance_scaled'] = (output_df['Distance'] - min_val) / (max_val - min_val)
output_df['Condition'] ='kf2vec_off_emb'

output_df.to_csv(os.path.join(tree_path, "distances_kf2vec_emb_R.txt"), sep='\t', index=False)
print("Saved distances")

Mean within-sample distance: 51.57341237498868
Mean across-sample distance: 338.3755017014051
Saved distances


In [51]:
# Assume dist_df is your distance DataFrame with full IDs
full_ids = dist_df.index.tolist()
n = len(full_ids)

# Generate unique 10-character PHYLIP IDs: e.g., ID000001, ID000002, ...
phylip_ids = [f"ID{str(i+1).zfill(6)}"[:10] for i in range(n)]

# Create mapping table
mapping_df = pd.DataFrame({
    "phylip_id": phylip_ids,
    "full_id": full_ids
})

# Save mapping to CSV
mapping_df.to_csv(os.path.join(fastme_folder, "phylip_name_mapping.csv"), index=False)


In [52]:
mapping_df

,phylip_id,full_id
0,ID000001,G900094975.part_NZ_FMAU01000004.1
1,ID000002,G002006995.part_NZ_MRUL01000011.1
2,ID000003,G001916215.part_MNRZ01000001.1
3,ID000004,G000336655.part_NZ_AOLZ01000076.1
4,ID000005,G000768875.part_NZ_JRAO01000018.1
...,...,...
2257,ID002258,G001312505.part_NZ_BBDW01000009.1
2258,ID002259,G001767605.part_MELM01000044.1
2259,ID002260,G000436935.part_FR891761.1
2260,ID002261,G001570425.part_NZ_BCSX01000018.1


In [53]:
# Apply new PHYLIP-safe IDs to distance DataFrame
dist_df_phylip = dist_df.copy()
dist_df_phylip.index = phylip_ids
dist_df_phylip.columns = phylip_ids

# Write PHYLIP distance matrix
with open(os.path.join(fastme_folder, "pairwise_distances_kf2vec_emb_l2.phy"), "w") as f:
    f.write(f"{n}\n")  # first line: number of taxa
    for i, taxon in enumerate(phylip_ids):
        row = " ".join(f"{dist_df_phylip.iloc[i, j]:.6f}" for j in range(n))
        f.write(f"{taxon.ljust(10)} {row}\n")


with open("pairwise_distances_kf2vec_emb_l2.phy", "w") as f:
    n = dist_df.shape[0]
    f.write(f"{n}\n")
    for i, taxon in enumerate(dist_df.index):
        # PHYLIP expects 10-character taxon names
        taxon_name = taxon[:10].ljust(10)
        row = " ".join(f"{x:.6f}" for x in dist_df.iloc[i])
        f.write(f"{taxon_name} {row}\n")


In [54]:
tree_path = fastme_folder

#grep -v "^[# ]" combined_distances.newick | grep -v "^$" > cleaned_tree.nwk
#sed -E 's/\[.*\]//g' cleaned_tree.nwk > cleaned_no_meta.nwk

tree_file = "kf2vec_fastme_tree_GlobalModel.nwk"

tree = treeswift.read_tree_newick(os.path.join(tree_path, tree_file))

# Get all leaf labels
leaves = [node.label for node in tree.traverse_leaves()]


In [55]:
# Load mapping file into dict
mapping = {}
with open(os.path.join(fastme_folder, "phylip_name_mapping.csv"), newline="") as f:
    reader = csv.DictReader(f)  # uses the header row automatically
    for row in reader:
        mapping[row["phylip_id"]] = row["full_id"]

print(mapping)
        

# Replace leaf labels
for node in tree.traverse_leaves():
    if node.label in mapping:
        node.label = mapping[node.label]

{'ID000001': 'G900094975.part_NZ_FMAU01000004.1', 'ID000002': 'G002006995.part_NZ_MRUL01000011.1', 'ID000003': 'G001916215.part_MNRZ01000001.1', 'ID000004': 'G000336655.part_NZ_AOLZ01000076.1', 'ID000005': 'G000768875.part_NZ_JRAO01000018.1', 'ID000006': 'G001424305.part_NZ_LMPD01000003.1', 'ID000007': 'G001074355.part_NZ_JUTO01000028.1', 'ID000008': 'G900107575.part_FNQX01000008.1', 'ID000009': 'G001649505.part_NZ_LVYG01000003.1', 'ID000010': 'G000382265.part_NZ_KB907974.1', 'ID000011': 'G001769705.part_MEPF01000028.1', 'ID000012': 'G000173655.part_NZ_ABZU01000009.1', 'ID000013': 'G000519205.part_NZ_JAFB01000031.1', 'ID000014': 'G000830925.part_NZ_CP006949.1', 'ID000015': 'G000429865.part_NZ_KE386978.1', 'ID000016': 'G000445105.part_NZ_ATDP01000103.1', 'ID000017': 'G900115085.part_FOVF01000007.1', 'ID000018': 'G900107845.part_FNRM01000001.1', 'ID000019': 'G000337675.part_NZ_AOIB01000043.1', 'ID000020': 'G001005215.part_NZ_LBNQ01000023.1', 'ID000021': 'G000482805.part_NZ_KI519505.1', '

In [56]:
# Get all leaf labels
leaves = [node.label for node in tree.traverse_leaves()]

In [57]:
leaves[:5]

['G000300115.part_NZ_JH930371.1',
 'G000300115.part_NZ_JH930370.1',
 'G000317125.part_NC_019695.1',
 'G000317125.part_NC_019699.1',
 'G001548375.part_NZ_AP017296.1']

In [58]:
M = tree.distance_matrix(leaf_labels=True)

In [59]:
# Build pandas DataFrame
leaves = list(M.keys())
df = pd.DataFrame(index=leaves, columns=leaves, dtype=float)

for i in leaves:
    for j in leaves:
        if i == j:
            df.loc[i, j] = 0.0   # distance to self
        else:
            df.loc[i, j] = M[i][j]

# Optional: save to CSV/TSV
df.to_csv(os.path.join(tree_path, "pairwise_distances_kf2vec_after_fastme.tsv"), sep="\t")

In [60]:
import pandas as pd
from itertools import combinations, product

# df is your pairwise distance matrix DataFrame
# Leaves labeled like "sample1.part_001", "sample2.part_010", etc.

# --- Step 1: group contigs by sample ID ---
sample_to_contigs = {}
for leaf in df.index:
    sample_id = leaf.split(".part")[0]  # extract sample ID
    sample_to_contigs.setdefault(sample_id, []).append(leaf)

# --- Step 2: compute within-sample distances ---
within_distances = []
records = []
for sample, contigs in sample_to_contigs.items():
    if len(contigs) < 2:
        continue  # no pairs to compute
    # all pairwise combinations of contigs in the same sample
    for a, b in combinations(contigs, 2):
        within_distances.append(df.loc[a, b])
        records.append({
            'Contig1': a,
            'Contig2': b,
            'Distance': df.loc[a, b],
            'Type': 'Within',
            'Sample1': sample,
            'Sample2': sample
        })

mean_within = sum(within_distances) / len(within_distances)
print("Mean within-sample distance:", mean_within)

# --- Step 3: compute across-sample distances ---
across_distances = []
samples = list(sample_to_contigs.keys())
for i in range(len(samples)):
    for j in range(i + 1, len(samples)):
        contigs_i = sample_to_contigs[samples[i]]
        contigs_j = sample_to_contigs[samples[j]]
        # all pairs between two different samples
        for a, b in product(contigs_i, contigs_j):
            across_distances.append(df.loc[a, b])
            records.append({
                'Contig1': a,
                'Contig2': b,
                'Distance': df.loc[a, b],
                'Type': 'Across',
                'Sample1': samples[i],
                'Sample2': samples[j]
            })

mean_across = sum(across_distances) / len(across_distances)
print("Mean across-sample distance:", mean_across)

# --- Step 4: save to text file ---
output_df = pd.DataFrame(records)

all_distances = output_df['Distance'].values
min_val = all_distances.min()
max_val = all_distances.max()
output_df['Distance_scaled'] = (output_df['Distance'] - min_val) / (max_val - min_val)
output_df['Condition'] ='kf2vec_after_fastme'

output_df.to_csv(os.path.join(tree_path, "distances_kf2vec_fastme_R.txt"), sep='\t', index=False)
print("Saved distances distances_kf2vec_placement.txt")


Mean within-sample distance: 59.54438767784314
Mean across-sample distance: 340.7975595578033
Saved distances distances_kf2vec_placement.txt
